# Probleme N°2: Prédiction de la DEMANDE taxi par zone & par heure

## 🎯 Objectif Business

👉 Prédire le nombre de courses à venirpar zone × heure pour :

- 🚕 repositionner les chauffeurs

- 📈 anticiper les pics de demande

- 💰 activer le surge pricing

- 🧠 aide à la décision opérationnelle

In [22]:
from pyspark.sql import functions as F

trips = spark.table("iceberg.silver.trips_complete")

hourly_demand = (
    trips
    .withColumn("hour_ts", F.date_trunc("hour", "tpep_pickup_datetime"))
    .groupBy("pickup_zone", "hour_ts")
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("trip_duration_minutes").alias("avg_duration"),
        F.avg("trip_distance").alias("avg_distance")
    )
)


In [23]:
from pyspark.sql.window import Window

w = Window.partitionBy("pickup_zone").orderBy("hour_ts")

features = (
    hourly_demand
    .withColumn("trip_count_next_hour", F.lead("trip_count").over(w))
    .filter("trip_count_next_hour IS NOT NULL")
)


In [24]:
features = (
    features
    .withColumn("hour", F.hour("hour_ts"))
    .withColumn("day_of_week", F.dayofweek("hour_ts"))
    .withColumn("is_weekend", F.col("day_of_week").isin(1,7))
)


In [25]:
weather = spark.table("iceberg.gold.weather_impact")

In [26]:
weather.limit(5).toPandas()

,pickup_date,tpep_pickup_datetime,is_rainy,is_cold,total_trips,avg_distance,avg_duration,avg_fare,avg_speed,avg_tip_pct,avg_temp,avg_rhum,avg_prcp,avg_wspd,avg_pres,weather_condition,year,month
0,2025-10-01,2025-10-01 00:31:03,False,False,2,1.44,6.483333,17.73,16.591712,19.871795,21.0,66.0,0.0,15.0,1018.0,dry_warm,2025,10
1,2025-10-01,2025-10-01 00:43:53,False,False,1,2.79,7.783333,21.33,21.507495,13.033286,21.0,66.0,0.0,15.0,1018.0,dry_warm,2025,10
2,2025-10-01,2025-10-01 00:37:41,False,False,1,5.34,11.083333,29.15,28.908271,10.291595,21.0,66.0,0.0,15.0,1018.0,dry_warm,2025,10
3,2025-10-01,2025-10-01 01:32:08,False,False,1,0.74,4.100000,14.70,10.829268,16.666667,19.0,68.0,0.0,11.0,1018.0,dry_warm,2025,10
4,2025-10-01,2025-10-01 03:42:44,False,False,1,2.59,10.250000,19.20,15.160976,0.000000,19.0,63.0,0.0,15.0,1018.0,dry_warm,2025,10


In [27]:
weather.printSchema()

root
 |-- pickup_date: date (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- is_rainy: boolean (nullable = true)
 |-- is_cold: boolean (nullable = true)
 |-- total_trips: long (nullable = true)
 |-- avg_distance: double (nullable = true)
 |-- avg_duration: double (nullable = true)
 |-- avg_fare: double (nullable = true)
 |-- avg_speed: double (nullable = true)
 |-- avg_tip_pct: double (nullable = true)
 |-- avg_temp: double (nullable = true)
 |-- avg_rhum: double (nullable = true)
 |-- avg_prcp: double (nullable = true)
 |-- avg_wspd: double (nullable = true)
 |-- avg_pres: double (nullable = true)
 |-- weather_condition: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)



In [32]:
features.printSchema()
features.limit(5).toPandas()

root
 |-- pickup_zone: string (nullable = true)
 |-- hour_ts: timestamp (nullable = true)
 |-- trip_count: long (nullable = false)
 |-- avg_duration: double (nullable = true)
 |-- avg_distance: double (nullable = true)
 |-- trip_count_next_hour: long (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)



,pickup_zone,hour_ts,trip_count,avg_duration,avg_distance,trip_count_next_hour,hour,day_of_week,is_weekend
0,None,2024-01-01 00:00:00,13,20.510256,3.379231,16,0,2,False
1,None,2024-01-01 01:00:00,16,15.281250,3.946875,19,1,2,False
2,None,2024-01-01 02:00:00,19,13.134211,2.246842,19,2,2,False
3,None,2024-01-01 03:00:00,19,14.529825,3.633684,14,3,2,False
4,None,2024-01-01 04:00:00,14,13.751190,4.855714,6,4,2,False


In [28]:
weather = spark.table("iceberg.gold.weather_impact") \
    .select(
        "pickup_date",
        F.col("avg_duration").alias("weather_avg_duration"),
        "avg_temp",
        "avg_prcp"
    )


In [29]:
features = (
    features
    .join(
        weather,
        features.pickup_date == weather.pickup_date,
        "left"
    )
)


AttributeError: 'DataFrame' object has no attribute 'pickup_date'

In [39]:
features.limit(5).toPandas()

,pickup_zone,hour_ts,trip_count,avg_duration,avg_distance,trip_count_next_hour,hour,day_of_week,is_weekend,pickup_date,tpep_pickup_datetime,is_rainy,is_cold,total_trips,avg_distance,avg_duration,avg_fare,avg_speed,avg_tip_pct,avg_temp,avg_rhum,avg_prcp,avg_wspd,avg_pres,weather_condition,year,month,pickup_date,weather_avg_duration,avg_temp,avg_prcp
0,Allerton/Pelham Gardens,2024-01-02 06:00:00,1,59.65,18.5,1,6,3,False,2024-01-02,2024-01-02 06:16:57,False,True,1,13.5,60.416667,45.0,13.406897,0.0,0.0,69.0,0.0,13.0,1018.0,dry_cold,2024,1,2024-01-02,14.108333,3.0,0.0
1,Allerton/Pelham Gardens,2024-01-02 06:00:00,1,59.65,18.5,1,6,3,False,2024-01-02,2024-01-02 06:16:57,False,True,1,13.5,60.416667,45.0,13.406897,0.0,0.0,69.0,0.0,13.0,1018.0,dry_cold,2024,1,2024-01-02,16.016667,2.0,0.0
2,Allerton/Pelham Gardens,2024-01-02 06:00:00,1,59.65,18.5,1,6,3,False,2024-01-02,2024-01-02 06:16:57,False,True,1,13.5,60.416667,45.0,13.406897,0.0,0.0,69.0,0.0,13.0,1018.0,dry_cold,2024,1,2024-01-02,2.383333,2.0,0.0
3,Allerton/Pelham Gardens,2024-01-02 06:00:00,1,59.65,18.5,1,6,3,False,2024-01-02,2024-01-02 06:16:57,False,True,1,13.5,60.416667,45.0,13.406897,0.0,0.0,69.0,0.0,13.0,1018.0,dry_cold,2024,1,2024-01-02,11.550000,2.0,0.0
4,Allerton/Pelham Gardens,2024-01-02 06:00:00,1,59.65,18.5,1,6,3,False,2024-01-02,2024-01-02 06:16:57,False,True,1,13.5,60.416667,45.0,13.406897,0.0,0.0,69.0,0.0,13.0,1018.0,dry_cold,2024,1,2024-01-02,25.983333,3.0,0.0


In [42]:
train = features.filter("hour_ts < '2025-01-01'")
test  = features.filter("hour_ts >= '2025-01-01'")


In [44]:
print(f"📊 Train: {train.count():,}")
print(f"📊 Test : {test.count():,}")

ConnectionRefusedError: [Errno 111] Connection refused

In [36]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline

cat_cols = ["pickup_zone"]
num_cols = [
    "trip_count",
    "avg_duration",
    "avg_distance",
    "hour",
    "day_of_week",
    "avg_temp",
    "avg_prcp"
]

indexer = StringIndexer(
    inputCol="pickup_zone",
    outputCol="pickup_zone_idx",
    handleInvalid="keep"
)

assembler = VectorAssembler(
    inputCols=num_cols + ["pickup_zone_idx"],
    outputCol="features",
    handleInvalid="keep"
)

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="trip_count_next_hour",
    numTrees=50,
    maxDepth=10
)

pipeline = Pipeline(stages=[indexer, assembler, rf])
model = pipeline.fit(train)


AnalysisException: [AMBIGUOUS_REFERENCE] Reference `avg_duration` is ambiguous, could be: [`avg_duration`, `iceberg`.`gold`.`weather_impact`.`avg_duration`].

In [ ]:
daily_metrics=